In [3]:
# ============================================================
# Cell 1 — Setup: mount Drive, imports, paths
# ============================================================
import os, math, time, random, warnings
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import xarray as xr
warnings.filterwarnings('ignore')

# --- Drive mount ---
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = Path('/content/drive/My Drive/Python Codes')
SEIS_DIR   = DRIVE_ROOT / 'SeisSavanna'
RUN_DIR    = DRIVE_ROOT / 'SJEPA_runs'
RUN_DIR.mkdir(parents=True, exist_ok=True)

SPEC_FILE  = SEIS_DIR / 'dset_allspec_150_spectrograms.nc'
assert SPEC_FILE.exists(), f'Spectrogram file not found: {SPEC_FILE}'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True

import xarray as xr

# Open just the metadata — does not download the full file
# First copy to Drive via the Dropbox link
# OR check what variables exist in your existing file
ds_check = xr.open_dataset(
    '/content/drive/My Drive/Python Codes/SeisSavanna/'
    'dset_allspec_150_spectrograms.nc'
)
print(list(ds_check.variables))
print(ds_check)

# --- Logging ---
RUN_NAME = f'sjepa_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
LOG_FILE = RUN_DIR / f'{RUN_NAME}.log'

def log(msg):
    ts = datetime.now().strftime('%H:%M:%S')
    line = f'[{ts}] {msg}'
    print(line)
    with open(LOG_FILE, 'a') as f:
        f.write(line + '\n')

log(f'Device: {DEVICE}')
if torch.cuda.is_available():
    log(f'GPU: {torch.cuda.get_device_name(0)}')
log(f'Spec file: {SPEC_FILE} ({SPEC_FILE.stat().st_size/1e9:.2f} GB)')

Mounted at /content/drive
['mel_spectr', 'class', 'distance', 'seis']
<xarray.Dataset> Size: 14GB
Dimensions:     (traces: 70637, side: 128, comp: 3)
Dimensions without coordinates: traces, side, comp
Data variables:
    mel_spectr  (traces, side, side, comp) float32 14GB ...
    class       (traces) int32 283kB ...
    distance    (traces) float32 283kB ...
    seis        (traces) int64 565kB ...
[16:26:58] Device: cuda
[16:26:58] GPU: NVIDIA A100-SXM4-80GB
[16:26:58] Spec file: /content/drive/My Drive/Python Codes/SeisSavanna/dset_allspec_150_spectrograms.nc (13.89 GB)


In [4]:
import subprocess

URL = 'https://www.dropbox.com/scl/fo/avwuo0jx9tjp6167auk2b/APb9q6h8NCNX71b5F0ENiyA/dset_allspec_150/dset_allspec_150_chunks_clean.nc?rlkey=xunpptuv7xn396ylyuiw3q8xw&dl=1'

OUTPUT = '/content/drive/My Drive/Python Codes/SeisSavanna/dset_allspec_150_chunks_clean.nc'

cmd = [
    'wget',
    '-c',
    '--tries=0',
    '--timeout=30',
    '--waitretry=10',
    '--read-timeout=30',
    '-O', OUTPUT,
    URL
]

print('Starting download — saves directly to Drive, auto-resumes if interrupted...')
subprocess.run(cmd)
print('Done.')

Starting download — saves directly to Drive, auto-resumes if interrupted...
Done.


In [5]:
# ============================================================
# Cell 2 — Run configuration (edit here only)
# ============================================================

# --- Class name lookup (used by Cell 3 and Cell 11) ---
CLASS_NAMES = {
    0:'noise',      1:'HUMAN',      2:'ELEPHANT',  3:'SUIDAE',
    4:'FELIDAE',    5:'NUMIDIDAE',  6:'BOVIDAE',   7:'GIRAFFIDAE',
    8:'HYAENIDAE',  9:'LEPORIDAE', 10:'EQUIDAE',  11:'MUSTELIDAE'
}

# --- Data ---
VIABLE_CLASSES   = [0, 1, 2, 5, 6, 9]
TRAIN_FRAC       = 0.75
VAL_FRAC         = 0.10
TEST_FRAC        = 0.15

# --- Spectrogram / input ---
IMG_H            = 128
IMG_W            = 128
N_COMP           = 3
AUGMENT_LOWFREQ  = True
N_LOWFREQ_BINS   = 32
IN_CHANNELS      = N_COMP * 2 if AUGMENT_LOWFREQ else N_COMP

# --- Patching ---
PATCH_H          = 16
PATCH_W          = 16
STRIDE_H         = 8
STRIDE_W         = 8

# --- Masking ---
MASK_RATIO       = 0.50
MASK_SCALES      = [(2,2), (4,4), (1,1)]

# --- Context encoder ---
D_MODEL          = 256
N_HEADS          = 8
N_LAYERS         = 4
MLP_RATIO        = 4
DROPOUT          = 0.1

# --- Predictor ---
D_PRED           = 128
N_PRED_LAYERS    = 2

# --- Affine frequency target ---
USE_AFFINE_TARGET = True
AFFINE_MU_REC    = 0.0

# --- Structured Bayesian GMM ---
K_TIGHT          = 60
K_MEDIUM         = 100
K_LOOSE          = 40
K_TOTAL          = K_TIGHT + K_MEDIUM + K_LOOSE
ALPHA_0          = 1.0 / K_TOTAL
KAPPA_0          = 1.0
A_0              = 2.0
B0_TIGHT_FRAC    = 0.05
B0_MEDIUM_FRAC   = 0.20
B0_LOOSE_FRAC    = 2.00
LAMBDA_GMM       = 0.0

# --- Training ---
BATCH_SIZE       = 64
LR               = 3e-4
WEIGHT_DECAY     = 1e-4
WARMUP_EPOCHS    = 0
TOTAL_EPOCHS     = 100
SAVE_EVERY       = 10
GRAD_CLIP        = 1.0

# --- Evaluation ---
KNN_K            = 5
EVAL_EVERY       = 5

log(f'Run: {RUN_NAME}')
log(f'Viable classes: {VIABLE_CLASSES}')
log(f'Input channels: {IN_CHANNELS}  (augmented low-freq: {AUGMENT_LOWFREQ})')
log(f'Patch grid: {((IMG_H-PATCH_H)//STRIDE_H+1)} x '
    f'{((IMG_W-PATCH_W)//STRIDE_W+1)}')
log(f'GMM: K={K_TOTAL} '
    f'(tight={K_TIGHT}, medium={K_MEDIUM}, loose={K_LOOSE})')
log(f'CLASS_NAMES defined: {list(CLASS_NAMES.values())}')

[16:27:02] Run: sjepa_20260613_162658
[16:27:02] Viable classes: [0, 1, 2, 5, 6, 9]
[16:27:02] Input channels: 6  (augmented low-freq: True)
[16:27:02] Patch grid: 15 x 15
[16:27:02] GMM: K=200 (tight=60, medium=100, loose=40)
[16:27:02] CLASS_NAMES defined: ['noise', 'HUMAN', 'ELEPHANT', 'SUIDAE', 'FELIDAE', 'NUMIDIDAE', 'BOVIDAE', 'GIRAFFIDAE', 'HYAENIDAE', 'LEPORIDAE', 'EQUIDAE', 'MUSTELIDAE']


In [6]:
# ============================================================
# Cell 3 — Data loading (smart cache, fast reload, no RAM crash)
# ============================================================
import pandas as pd
import os

ds  = xr.open_dataset(str(SPEC_FILE))
cls = ds['class'].values
seis_ids = ds['seis'].values    # seismometer IDs per trace

# --- Show what is in the SeisSavanna folder ---
log('SeisSavanna folder contents:')
for f in sorted(os.listdir(str(SEIS_DIR))):
    p = os.path.join(str(SEIS_DIR), f)
    log(f'  {f}  ({os.path.getsize(p)/1e6:.1f} MB)')

# --- Find CSV ---
CSV_CANDIDATES = [
    SEIS_DIR / 'dset_allspec_150_chunks.csv',
    SEIS_DIR / 'chunks.csv',
    SEIS_DIR / 'dset_allspec_150' / 'dset_allspec_150_chunks.csv',
    DRIVE_ROOT / 'SeisSavanna' / 'dset_allspec_150_chunks.csv',
]
CSV_FILE = None
for candidate in CSV_CANDIDATES:
    if Path(candidate).exists():
        CSV_FILE = Path(candidate)
        log(f'CSV found: {CSV_FILE}')
        break
if CSV_FILE is None:
    log('CSV not found — using sample-level splits (no event grouping)')

# --- Load metadata ---
if CSV_FILE is not None:
    meta = pd.read_csv(str(CSV_FILE))
    log(f'CSV loaded: {len(meta)} rows  '
        f'{meta["photo_name"].nunique()} unique events')
    # The NetCDF has 70,637 traces; CSV has 82,593 rows (more seismometers)
    # Align by position — take first len(cls) rows matching NetCDF order
    photo_names = meta['photo_name'].values[:len(cls)]
    log(f'  Aligned to {len(cls)} NetCDF traces')
else:
    photo_names = np.arange(len(cls)).astype(str)

# --- Filter to viable classes ---
mask       = np.isin(cls, VIABLE_CLASSES)
indices    = np.where(mask)[0]
cls_filt   = cls[indices]
photo_filt = (photo_names[mask]
              if len(photo_names) >= len(cls)
              else np.arange(len(indices)).astype(str))

log(f'Viable samples: {len(indices)} / {len(cls)} total')
for c in VIABLE_CLASSES:
    n = int((cls_filt == c).sum())
    log(f'  class {c:2d} ({CLASS_NAMES.get(c,"?"):12s}): {n:6d}')

# --- Event-grouped train / val / test split ---
unique_events = np.unique(photo_filt)
np.random.seed(42)
np.random.shuffle(unique_events)
n_tr = int(TRAIN_FRAC * len(unique_events))
n_va = int(VAL_FRAC   * len(unique_events))
train_events = set(unique_events[:n_tr])
val_events   = set(unique_events[n_tr:n_tr+n_va])
test_events  = set(unique_events[n_tr+n_va:])

train_idx = [i for i, e in zip(range(len(indices)), photo_filt)
             if e in train_events]
val_idx   = [i for i, e in zip(range(len(indices)), photo_filt)
             if e in val_events]
test_idx  = [i for i, e in zip(range(len(indices)), photo_filt)
             if e in test_events]

log(f'Split: train={len(train_idx)}  '
    f'val={len(val_idx)}  test={len(test_idx)}')
log(f'  Using {"event-grouped" if CSV_FILE else "sample-level"} splits')

# --- Normalisation stats ---
np.random.seed(0)
sample_pos  = np.random.choice(train_idx, min(500, len(train_idx)),
                               replace=False)
sample_data = ds['mel_spectr'][indices[sample_pos]].values
NORM_MEAN   = sample_data.mean(axis=(0, 1, 2))
NORM_STD    = sample_data.std(axis=(0, 1, 2)) + 1e-6
log(f'Norm mean: {NORM_MEAN.round(3)}  std: {NORM_STD.round(3)}')

# --- SMART CACHE with save/reload ---
CACHE_PATH      = SEIS_DIR / 'smart_cache.npz'
MAX_HUMAN_CACHE = 30000

if CACHE_PATH.exists():
    log(f'Loading pre-saved cache from Drive...')
    t0    = time.time()
    saved = np.load(str(CACHE_PATH))
    cache_data      = saved['data']
    cache_local_idx = saved['local_idx']
    cache_map       = {int(li): ci
                       for ci, li in enumerate(cache_local_idx)}
    log(f'Cache loaded in {time.time()-t0:.1f}s:  '
        f'{len(cache_local_idx)} samples  '
        f'{cache_data.nbytes/1e9:.2f} GB')
else:
    log(f'Building smart cache '
        f'(all rare classes + {MAX_HUMAN_CACHE} human)...')
    t0 = time.time()
    cache_local_idx = []
    human_count     = 0
    for i in range(len(indices)):
        c = int(cls_filt[i])
        if c == 1:
            if human_count < MAX_HUMAN_CACHE:
                cache_local_idx.append(i)
                human_count += 1
        else:
            cache_local_idx.append(i)
    cache_local_idx = np.array(cache_local_idx)
    cache_global    = indices[cache_local_idx]
    log(f'  Fetching {len(cache_local_idx)} spectrograms from NetCDF...')
    cache_data      = ds['mel_spectr'][cache_global].values
    cache_map       = {int(li): ci
                       for ci, li in enumerate(cache_local_idx)}
    ram_gb  = cache_data.nbytes / 1e9
    n_rare  = len(cache_local_idx) - human_count
    log(f'Cache built in {time.time()-t0:.1f}s:  '
        f'{len(cache_local_idx)} samples  '
        f'{ram_gb:.2f} GB  '
        f'({human_count} human + {n_rare} rare)')
    log('Saving cache to Drive...')
    t1 = time.time()
    np.savez_compressed(str(CACHE_PATH),
                        data=cache_data,
                        local_idx=cache_local_idx)
    log(f'Cache saved in {time.time()-t1:.1f}s: {CACHE_PATH.name}')


class SeisDataset(Dataset):
    def __init__(self, local_indices, augment_lowfreq=True):
        self.local_idx = list(local_indices)
        self.augment   = augment_lowfreq

    def __len__(self):
        return len(self.local_idx)

    def __getitem__(self, i):
        li = int(self.local_idx[i])
        if li in cache_map:
            x = cache_data[cache_map[li]].copy()
        else:
            x = ds['mel_spectr'][indices[li]].values
        x = (x - NORM_MEAN) / NORM_STD
        if self.augment:
            low   = x[:N_LOWFREQ_BINS, :, :]
            low_t = (torch.from_numpy(low)
                     .permute(2, 0, 1)
                     .unsqueeze(0)
                     .float())
            low_u = F.interpolate(low_t,
                                  size=(IMG_H, IMG_W),
                                  mode='bilinear',
                                  align_corners=False)
            low_u = low_u.squeeze(0).permute(1, 2, 0).numpy()
            x     = np.concatenate([x, low_u], axis=-1)
        x = torch.from_numpy(x).permute(2, 0, 1).float()
        return x, int(cls_filt[li])


# ── Condition D: Unsupervised geometric sampler ───────────
WEIGHT_FILE = SEIS_DIR / 'sampling_weights.npy'
if WEIGHT_FILE.exists():
    all_weights_loaded = np.load(str(WEIGHT_FILE))
    train_weights      = all_weights_loaded[train_idx].astype(np.float64)
    sampler = WeightedRandomSampler(
        train_weights.tolist(),
        num_samples=len(train_idx),
        replacement=True
    )
    log(f'Condition D: geometric sampler loaded from Drive')
    log(f'  Weight range: [{train_weights.min():.2f}, '
        f'{train_weights.max():.2f}]')
else:
    log('WARNING: sampling_weights.npy not found — using uniform sampler')
    sampler = None

train_ds = SeisDataset(train_idx, AUGMENT_LOWFREQ)
val_ds   = SeisDataset(val_idx,   AUGMENT_LOWFREQ)
test_ds  = SeisDataset(test_idx,  AUGMENT_LOWFREQ)

if sampler is not None:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              sampler=sampler,
                              num_workers=0, pin_memory=True,
                              drop_last=True)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True,
                              num_workers=0, pin_memory=True,
                              drop_last=True)

val_loader  = DataLoader(val_ds,  batch_size=BATCH_SIZE,
                         shuffle=False,
                         num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE,
                         shuffle=False,
                         num_workers=0, pin_memory=True)

log(f'  train={len(train_ds):6d}  {len(train_loader):4d} batches/epoch')
log(f'  val  ={len(val_ds):6d}')
log(f'  test ={len(test_ds):6d}')
log(f'  batch_size={BATCH_SIZE}  num_workers=0')
log(f'Cell 3 complete.')

[16:27:04] SeisSavanna folder contents:
[16:27:04]   S_JEPA_V3_SSL_HMM.ipynb  (0.1 MB)
[16:27:04]   cluster_labels.npy  (0.3 MB)
[16:27:04]   dset_allspec_150_chunks.csv  (6.5 MB)
[16:27:04]   dset_allspec_150_chunks_clean.nc  (29513.0 MB)
[16:27:04]   dset_allspec_150_spectrograms.nc  (13888.9 MB)
[16:27:04]   projected_specs.npy  (5.6 MB)
[16:27:04]   sampling_weights.npy  (0.3 MB)
[16:27:04]   smart_cache.npz  (5986.4 MB)
[16:27:04] CSV found: /content/drive/My Drive/Python Codes/SeisSavanna/dset_allspec_150_chunks.csv
[16:27:05] CSV loaded: 82593 rows  9263 unique events
[16:27:05]   Aligned to 70637 NetCDF traces
[16:27:05] Viable samples: 70308 / 70637 total
[16:27:05]   class  0 (noise       ):   2073
[16:27:05]   class  1 (HUMAN       ):  62493
[16:27:05]   class  2 (ELEPHANT    ):    164
[16:27:05]   class  5 (NUMIDIDAE   ):    769
[16:27:05]   class  6 (BOVIDAE     ):   4321
[16:27:05]   class  9 (LEPORIDAE   ):    488
[16:27:05] Split: train=53037  val=6942  test=10329
[16:2

In [22]:
# ============================================================
# Cell 2b — Unsupervised rare-sample detection via FuzzyART
# Fast Fuzzy ART vigilance sweep (artlib)
# 20 physically motivated features (10 raw + 10 mel)
# NO labels used in clustering — only in post-hoc diagnostic
# ============================================================
import numpy as np
import time
from pathlib import Path
from scipy import signal
from scipy.signal import hilbert, correlate, find_peaks
from scipy.ndimage import uniform_filter1d
from scipy.stats import kurtosis as scipy_kurtosis
from sklearn.preprocessing import MinMaxScaler, StandardScaler

try:
    from artlib import FuzzyART
    log('artlib ready')
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'artlib',
                    '--break-system-packages', '--quiet'],
                   capture_output=True)
    from artlib import FuzzyART
    log('artlib installed')

FEAT_FILE   = SEIS_DIR / 'projected_specs.npy'
SCORE_FILE  = SEIS_DIR / 'cluster_labels.npy'
WEIGHT_FILE = SEIS_DIR / 'sampling_weights.npy'
CLEAN_FILE  = SEIS_DIR / 'dset_allspec_150_chunks_clean.nc'

WEIGHT_CAP   = len(indices) / 10
WEIGHT_FLOOR = 1.0
N_total      = len(indices)
N_FEAT       = 20
FS           = 100.0

log(f'Cell 2b: N_total={N_total}  N_FEAT={N_FEAT}')
log(f'  WEIGHT_CAP={WEIGHT_CAP:.0f}  WEIGHT_FLOOR={WEIGHT_FLOOR}')
log(f'  FEAT_FILE exists: {FEAT_FILE.exists()}')
log(f'  SCORE_FILE exists: {SCORE_FILE.exists()}')
log(f'  WEIGHT_FILE exists: {WEIGHT_FILE.exists()}')


def extract_features_raw(z, n, e, dist):
    f   = np.zeros(10, dtype=np.float32)
    eps = 1e-30
    f[0] = float(np.abs(z).max() * dist**2)
    freqs, psd = signal.welch(z, fs=FS, nperseg=512,
                               noverlap=256)
    psd_norm = psd / (psd.sum() + eps)
    f[1] = float(-np.sum(psd_norm * np.log(psd_norm + eps)))
    cepstrum = np.abs(np.fft.irfft(np.log(psd + eps)))
    lo = int(0.5 * FS / 2)
    hi = min(int(3.0 * FS / 2), len(cepstrum)-1)
    if hi > lo:
        gr = cepstrum[lo:hi]
        f[2] = float(gr.max())
        f[3] = float(gr.argmax() + lo) / FS
    else:
        f[2] = f[3] = 0.0
    try:
        eigs = np.sort(np.linalg.eigvalsh(
            np.cov(np.stack([z, n, e]))))[::-1]
        eigs = np.abs(eigs) + eps
        f[4] = float(1.0 - eigs[1]/eigs[0])
        f[5] = float(1.0 - 2*eigs[2]/(eigs[0]+eigs[1]))
    except Exception:
        f[4] = f[5] = 0.0
    inst_freq = np.clip(
        np.diff(np.unwrap(np.angle(hilbert(z))))
        * FS / (2*np.pi), 0, FS/2)
    f[6] = float(inst_freq.mean())
    f[7] = float(inst_freq[-100:].mean() -
                 inst_freq[:100].mean())
    zn   = correlate(z-z.mean(), n-n.mean(), mode='full')
    f[8] = float((np.argmax(np.abs(zn)) - len(z)+1) / FS)
    f[9] = float(
        psd[(freqs>=1.0)&(freqs<2.0)].mean() /
        (psd[(freqs>=5.0)&(freqs<10.0)].mean()+eps))
    return f


def extract_features_mel(x):
    f   = np.zeros(10, dtype=np.float32)
    eps = 1e-8
    surf = x[:10,  :, 2].mean()
    body = x[10:32,:, 2].mean()
    f[0] = float(surf / (body+eps))
    z_e  = x[:32,  :, 2].mean()
    h_e  = (x[:32,:,0].mean() + x[:32,:,1].mean()) / 2.0
    f[1] = float(z_e / (h_e+eps))
    f[2] = float(np.corrcoef(x[:32,:,2].flatten(),
                              x[:32,:,1].flatten())[0,1])
    f[3] = float(np.corrcoef(x[:32,:,2].flatten(),
                              x[:32,:,0].flatten())[0,1])
    envelope = x[:32,:,2].mean(axis=0)
    smooth   = uniform_filter1d(envelope, size=5)
    peaks, _ = find_peaks(smooth,
                           height=smooth.mean()+smooth.std(),
                           distance=10)
    if len(peaks) > 0:
        mp    = peaks[np.argmax(smooth[peaks])]
        thresh= smooth.mean()+0.3*(smooth[mp]-smooth.mean())
        after = smooth[mp:]
        de    = np.where(after < thresh)[0]
        f[4]  = float(de[0]) if len(de)>0 \
                else float(len(after))
    else:
        f[4] = 0.0
    f[5] = float(scipy_kurtosis(envelope))
    env_norm = envelope - envelope.mean()
    if env_norm.std() > 1e-6:
        ac       = np.correlate(env_norm, env_norm,
                                mode='full')
        ac       = ac[len(ac)//2:]
        ac       = ac / (ac[0]+eps)
        ac_range = ac[8:40]
        f[6]     = float(ac_range.max())
        f[7]     = float(ac_range.argmax()+8)
    else:
        f[6] = f[7] = 0.0
    f[8] = float(x[:32,:,2].mean() - x[32:,:,2].mean())
    f[9] = float(x[:32,:,2].mean() /
                 (x[32:,:,2].mean()+eps))
    return f


# ── Stage 2: Load or extract features ─────────────────────
if FEAT_FILE.exists():
    log('\nStage 2: Loading existing features from Drive...')
    t0           = time.time()
    features_all = np.load(str(FEAT_FILE))
    log(f'  Loaded in {time.time()-t0:.1f}s  '
        f'shape={features_all.shape}')
else:
    if not CLEAN_FILE.exists():
        raise FileNotFoundError(
            f'Clean NetCDF not found: {CLEAN_FILE}')
    import xarray as xr
    log('Opening clean NetCDF...')
    ds_clean   = xr.open_dataset(str(CLEAN_FILE))
    dist_clean = ds_clean['distance'].values
    log(f'Stage 2: Extracting {N_FEAT} features '
        f'from {N_total} spectrograms...')
    t0           = time.time()
    BATCH        = 256
    features_all = np.zeros((N_total, N_FEAT),
                            dtype=np.float32)
    for start in range(0, N_total, BATCH):
        end        = min(start+BATCH, N_total)
        mel_batch  = ds['mel_spectr'][
            indices[start:end]].values
        mel_batch  = (mel_batch-NORM_MEAN)/NORM_STD
        raw_batch  = ds_clean['chunk'][
            indices[start:end]].values
        dist_batch = dist_clean[indices[start:end]]
        for i in range(end-start):
            z    = raw_batch[i, 2, :]
            n    = raw_batch[i, 1, :]
            e    = raw_batch[i, 0, :]
            dist = float(dist_batch[i])
            features_all[start+i] = np.concatenate([
                extract_features_raw(z, n, e, dist),
                extract_features_mel(mel_batch[i])])
        if start % 2000 == 0 or end == N_total:
            pct = 100*end/N_total
            eta = (time.time()-t0)/max(end,1)*(N_total-end)
            print(f'  {end}/{N_total} ({pct:.0f}%)  '
                  f'ETA={eta:.0f}s', flush=True)
    log(f'  Done in {time.time()-t0:.1f}s')
    scaler       = StandardScaler()
    features_all = scaler.fit_transform(
        features_all).astype(np.float32)
    features_all = np.nan_to_num(features_all, 0.0)
    np.save(str(FEAT_FILE), features_all)
    log('  Saved: projected_specs.npy')


# ── Stage 3: Scale to [0,1] for FuzzyART ──────────────────
log('\nStage 3: Scaling to [0,1] for FuzzyART...')
scaler_mm   = MinMaxScaler()
features_01 = scaler_mm.fit_transform(
    features_all).astype(np.float32)
log(f'  Done. shape={features_01.shape}')


# ── Stage 4: FuzzyART vigilance sweep ─────────────────────
# FuzzyART is O(n*K) — ~3 min per combination on 70k samples
# Higher rho = more clusters = more fine-grained
log('\nStage 4: FuzzyART vigilance sweep...')
log('  rho → n_clusters → ratio  (est. ~3 min each)')

sweep_rho = [0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

best_ratio  = -1.0
best_labels = None
best_sizes  = None
best_rho    = None

for rho in sweep_rho:
    try:
        t0    = time.time()
        model = FuzzyART(rho=rho, alpha=0.001, beta=1.0)
        X_prep = model.prepare_data(features_01)
        model.fit(X_prep)
        labels     = model.predict(X_prep).astype(np.int32)
        n_clusters = len(np.unique(labels))
        sizes      = np.bincount(labels,
                                 minlength=n_clusters)

        # Inverse-prior weights
        w = np.zeros(N_total, dtype=np.float32)
        for i in range(N_total):
            k    = labels[i]
            wi   = float(N_total) / float(sizes[k])
            w[i] = min(max(wi, WEIGHT_FLOOR), WEIGHT_CAP)

        # Post-hoc validation
        h_mean = w[cls_filt == 1].mean()
        e_mean = w[cls_filt == 2].mean()
        ratio  = e_mean / (h_mean + 1e-8)
        elapsed= time.time()-t0

        log(f'  rho={rho:.2f}  '
            f'clusters={n_clusters:6d}  '
            f'min={sizes.min():4d}  '
            f'max={sizes.max():6d}  '
            f'ratio={ratio:.3f}x  '
            f't={elapsed:.0f}s')

        if ratio > best_ratio:
            best_ratio  = ratio
            best_labels = labels.copy()
            best_sizes  = sizes.copy()
            best_rho    = rho

    except Exception as ex:
        log(f'  rho={rho:.2f} FAILED: {ex}')

log(f'\nBest: rho={best_rho}  ratio={best_ratio:.3f}x')


# ── Stage 5: Final sampling weights ───────────────────────
log('\nStage 5: Computing final sampling weights...')
all_weights = np.zeros(N_total, dtype=np.float32)
for i in range(N_total):
    k = best_labels[i]
    w = float(N_total) / float(best_sizes[k])
    all_weights[i] = min(max(w, WEIGHT_FLOOR), WEIGHT_CAP)

log(f'  Weight range: [{all_weights.min():.2f}, '
    f'{all_weights.max():.2f}]')

log('\nSaving to Drive...')
np.save(str(SCORE_FILE),  best_labels)
np.save(str(WEIGHT_FILE), all_weights)
log('  Saved: cluster_labels.npy  sampling_weights.npy')


# ── Stage 6: Validation diagnostic ────────────────────────
log('\n' + '='*55)
log('Validation diagnostic (post-hoc — not used in training)')
log('='*55)

n_clusters   = len(np.unique(best_labels))
human_weight = all_weights[cls_filt == 1].mean()
eleph_weight = all_weights[cls_filt == 2].mean()
ratio        = eleph_weight / human_weight

log(f'Best FuzzyART rho: {best_rho}')
log(f'n_clusters: {n_clusters}\n')

log(f'  {"Class":14s}  {"n":>6s}  '
    f'{"mean_weight":>12s}  '
    f'{"mean_cluster_sz":>16s}  '
    f'{"ratio_vs_human":>14s}')
log('  ' + '-'*68)

for c in VIABLE_CLASSES:
    mask = (cls_filt == c)
    if mask.sum() == 0:
        continue
    mw  = all_weights[mask].mean()
    mcs = best_sizes[best_labels[mask]].mean()
    r   = mw / human_weight
    log(f'  {CLASS_NAMES.get(c,"?"):14s}  '
        f'{mask.sum():6d}  '
        f'{mw:12.2f}  '
        f'{mcs:16.0f}  '
        f'{r:14.2f}x')

log(f'\nElephant/Human weight ratio: {ratio:.2f}x')
if ratio >= 3.0:
    log('>> PASS — proceed with Condition E training')
elif ratio >= 1.5:
    log(f'>> MARGINAL: {ratio:.2f}x — '
        f'modest improvement possible')
else:
    log(f'>> WEAK: {ratio:.2f}x — '
        f'effectively uniform sampling')

log('\nCell 2b complete.')

[18:13:45] artlib ready
[18:13:45] Cell 2b: N_total=70308  N_FEAT=20
[18:13:45]   WEIGHT_CAP=7031  WEIGHT_FLOOR=1.0
[18:13:45]   FEAT_FILE exists: True
[18:13:45]   SCORE_FILE exists: False
[18:13:45]   WEIGHT_FILE exists: False
[18:13:45] 
Stage 2: Loading existing features from Drive...
[18:13:45]   Loaded in 0.0s  shape=(70308, 20)
[18:13:45] 
Stage 3: Scaling to [0,1] for FuzzyART...
[18:13:45]   Done. shape=(70308, 20)
[18:13:45] 
Stage 4: FuzzyART vigilance sweep...
[18:13:45]   rho → n_clusters → ratio  (est. ~3 min each)
[18:13:56]   rho=0.60  clusters=    59  min=  10  max= 10034  ratio=2.458x  t=11s
[18:14:35]   rho=0.70  clusters=   281  min=   3  max=  3320  ratio=1.511x  t=40s
[18:16:02]   rho=0.75  clusters=   630  min=   3  max=  2238  ratio=1.738x  t=87s
[18:20:32]   rho=0.80  clusters=  1709  min=   2  max=   650  ratio=1.432x  t=270s


KeyboardInterrupt: 

In [8]:
# ============================================================
# Cell 4 — Affine frequency target module
# w_f * mel_f + b_f  (per frequency bin, 256 parameters total)
# Initialised at identity: w=1, b=0 → target = mel_patch at init
# ============================================================

class AffineFrequencyTarget(nn.Module):
    """
    Learned per-frequency affine transformation on mel-spectrogram patches.

    target_f = w_f * mel_f + b_f

    - w_f: learned gain per frequency bin (init=1, amplifies informative bands)
    - b_f: learned offset per frequency bin (init=0, corrects noise-floor bias)

    Biological analogy:
      w_f = position-dependent cochlear sensitivity (frequency-specific gain)
      b_f = rate-level adaptation (background noise-floor correction)

    Anti-collapse: affine transform of non-degenerate input cannot collapse
    to a constant. Reconstruction loss L_rec guards against w->0 drift.

    Parameters: 2 * n_freq_bins = 256 (negligible).
    """
    def __init__(self, n_freq_bins=128):
        super().__init__()
        self.w = nn.Parameter(torch.ones(n_freq_bins))
        self.b = nn.Parameter(torch.zeros(n_freq_bins))

    def forward(self, patch):
        """
        patch: (B, C, patch_h, patch_w) — mel-spectrogram patch
        Applies per-frequency (height) affine transform.
        w, b shape: (n_freq_bins,) -> broadcast over patch_w and C
        """
        # patch height = PATCH_H frequency bins
        # w, b: (PATCH_H,) -> (1, 1, PATCH_H, 1)
        w = self.w[:patch.shape[2]].view(1, 1, -1, 1)
        b = self.b[:patch.shape[2]].view(1, 1, -1, 1)
        return w * patch + b

    def reconstruction_loss(self, patch):
        """Prevents collapse: target must remain faithful to input."""
        return F.mse_loss(self.forward(patch), patch)

    def frequency_profile(self):
        """Diagnostic: plot learned w and b across frequency bins."""
        import matplotlib.pyplot as plt
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        w = self.w.detach().cpu().numpy()
        b = self.b.detach().cpu().numpy()
        ax1.plot(w); ax1.axhline(1.0, color='red', linestyle='--', alpha=0.5)
        ax1.set_title('Learned frequency gain w_f (init=1)')
        ax1.set_xlabel('Frequency bin'); ax1.set_ylabel('Gain')
        ax2.plot(b); ax2.axhline(0.0, color='red', linestyle='--', alpha=0.5)
        ax2.set_title('Learned frequency offset b_f (init=0)')
        ax2.set_xlabel('Frequency bin'); ax2.set_ylabel('Offset (dB)')
        plt.tight_layout(); plt.show()


affine_target = AffineFrequencyTarget(n_freq_bins=IMG_H).to(DEVICE)
log(f'Affine target params: {sum(p.numel() for p in affine_target.parameters())}')

[16:34:25] Affine target params: 256


In [9]:
# ============================================================
# Cell 5 — Structured Bayesian GMM with VB-EM
# Three component groups: tight / medium / loose prior on covariance
# All M-step updates are closed-form (no gradient, no learning rate)
# ============================================================

class StructuredBayesianGMM(nn.Module):
    """
    Variational Bayes GMM with structured covariance priors.

    Component groups:
      Tight  (K_tight):  small b_0 → small posterior covariance
                         → structural affinity for compact rare clusters
      Medium (K_medium): standard b_0 → general purpose
      Loose  (K_loose):  large b_0 → large posterior covariance
                         → absorbs diffuse majority structure

    Inference: VB-EM with Normal-InverseGamma conjugate priors.
    All M-step updates are analytical (no gradient).
    ELBO gradient backprops into the encoder through the E-step.

    Key property: alpha_0 > 0 is the formal rare-class protection.
    Even a component with N_k=0 retains pi_k >= alpha_0 / (K*alpha_0 + N)
    and cannot be eliminated.
    """
    def __init__(self, d_embed,
                 K_tight=60, K_medium=100, K_loose=40,
                 alpha_0=None, kappa_0=1.0, a_0=2.0,
                 b0_tight=0.01, b0_medium=0.10, b0_loose=1.00):
        super().__init__()
        self.K       = K_tight + K_medium + K_loose
        self.d       = d_embed
        self.K_tight  = K_tight
        self.K_medium = K_medium
        self.K_loose  = K_loose
        self.alpha_0  = alpha_0 if alpha_0 is not None else 1.0 / self.K
        self.kappa_0  = kappa_0
        self.a_0      = a_0

        # Build structured b_0: (K, d) — differs per component group
        b0_vec = torch.cat([
            torch.full((K_tight,),  b0_tight),
            torch.full((K_medium,), b0_medium),
            torch.full((K_loose,),  b0_loose),
        ])                                                   # (K,)
        b0_full = b0_vec.unsqueeze(1).expand(self.K, d_embed).clone()

        # Posterior parameters as buffers (updated by EM, not by optimizer)
        self.register_buffer('alpha', torch.full((self.K,), self.alpha_0))
        self.register_buffer('mu',    torch.zeros(self.K, d_embed))
        self.register_buffer('kappa', torch.ones(self.K))
        self.register_buffer('a',     torch.full((self.K, d_embed), a_0))
        self.register_buffer('b',     b0_full.clone())
        self.register_buffer('b0',    b0_full)          # keep prior fixed

        self.m_0 = None   # set from data on first batch

    def _psi(self):
        """Posterior mean of component variances: E[psi_k] = b_k / (a_k - 1)."""
        return self.b / (self.a - 1.0).clamp(min=1e-6)   # (K, d)

    @torch.no_grad()
    def set_b0_from_data(self, embeddings):
        """
        Set b_0 values adaptively from empirical embedding variance.
        Call once after warmup epochs, before GMM activation.
        b0_tight  = B0_TIGHT_FRAC  * empirical_var
        b0_medium = B0_MEDIUM_FRAC * empirical_var
        b0_loose  = B0_LOOSE_FRAC  * empirical_var
        """
        emp_var = embeddings.var(0).mean().item()
        b0_t = B0_TIGHT_FRAC  * emp_var
        b0_m = B0_MEDIUM_FRAC * emp_var
        b0_l = B0_LOOSE_FRAC  * emp_var
        b0_vec = torch.cat([
            torch.full((self.K_tight,),  b0_t),
            torch.full((self.K_medium,), b0_m),
            torch.full((self.K_loose,),  b0_l),
        ]).to(self.b.device)
        self.b0.copy_(b0_vec.unsqueeze(1).expand(self.K, self.d))
        self.b.copy_(self.b0)
        log(f'GMM b_0 set from data (emp_var={emp_var:.4f}): '
            f'tight={b0_t:.5f} medium={b0_m:.4f} loose={b0_l:.3f}')

    @torch.no_grad()
    def e_step(self, embeddings):
        """
        E-step: compute posterior responsibilities.
        embeddings: (B, d)
        Returns: r (B, K)
        """
        if self.m_0 is None:
            self.m_0 = embeddings.mean(0)

        psi     = self._psi()                                  # (K, d)
        log_pi  = (torch.digamma(self.alpha)
                   - torch.digamma(self.alpha.sum()))           # (K,)
        diff    = embeddings.unsqueeze(1) - self.mu.unsqueeze(0)  # (B,K,d)
        maha    = (diff**2 / psi.unsqueeze(0)).sum(-1)          # (B,K)
        log_det = psi.log().sum(-1)                             # (K,)
        log_lik = -0.5 * (self.d * math.log(2*math.pi)
                          + log_det.unsqueeze(0) + maha)        # (B,K)
        log_r   = log_pi.unsqueeze(0) + log_lik                 # (B,K)
        log_r   = log_r - torch.logsumexp(log_r, dim=1, keepdim=True)
        return log_r.exp()                                      # (B,K)

    @torch.no_grad()
    def m_step(self, embeddings, r):
        """
        M-step: closed-form Bayesian parameter updates.
        embeddings: (B, d)   r: (B, K)
        All updates are analytical — no gradient, no learning rate.
        """
        N_k   = r.sum(0)                                        # (K,)
        e_bar = (torch.einsum('bk,bd->kd', r, embeddings)
                 / (N_k.unsqueeze(1) + 1e-8))                  # (K,d)

        # Posterior Dirichlet — alpha_0 floor protects rare components
        self.alpha = self.alpha_0 + N_k                         # (K,)

        # Posterior mean — shrinks toward m_0 when N_k is small
        kappa_k    = self.kappa_0 + N_k                         # (K,)
        self.mu    = ((self.kappa_0 * self.m_0.unsqueeze(0)
                      + N_k.unsqueeze(1) * e_bar)
                      / kappa_k.unsqueeze(1))                   # (K,d)
        self.kappa = kappa_k

        # Posterior variance — structured b_0 creates group-specific floors
        diff_bar = e_bar - self.m_0.unsqueeze(0)                # (K,d)
        shrink   = ((self.kappa_0 * N_k / kappa_k)
                    .unsqueeze(1) * diff_bar**2)                 # (K,d)
        S_k      = torch.einsum(
            'bk,bkd->kd', r,
            (embeddings.unsqueeze(1)
             - e_bar.unsqueeze(0))**2)                          # (K,d)
        self.a = self.a_0 + N_k.unsqueeze(1) / 2               # (K,d)
        self.b = self.b0  + (S_k + shrink) / 2                 # (K,d)

    def elbo_loss(self, embeddings, r):
        """
        ELBO term that backprops into the encoder.
        Pushes encoder to produce well-separated embeddings
        under the mixture geometry.
        """
        psi     = self._psi().detach()                          # (K,d) no grad
        mu_det  = self.mu.detach()
        log_pi  = (torch.digamma(self.alpha.detach())
                   - torch.digamma(self.alpha.detach().sum()))
        diff    = embeddings.unsqueeze(1) - mu_det.unsqueeze(0)
        maha    = (diff**2 / psi.unsqueeze(0)).sum(-1)
        log_det = psi.log().sum(-1)
        log_lik = -0.5 * (self.d * math.log(2*math.pi)
                          + log_det.unsqueeze(0) + maha)
        log_r   = log_pi.unsqueeze(0) + log_lik
        log_r   = log_r - torch.logsumexp(log_r, dim=1, keepdim=True)
        # Expected log-likelihood — maximised when embeddings fit components well
        return -(r.detach() * log_lik).sum(-1).mean()

    def component_summary(self):
        """Diagnostic: active components and mean variance per group."""
        pi       = (self.alpha / self.alpha.sum()).cpu()
        psi_mean = self._psi().mean(-1).cpu()
        t_end    = self.K_tight
        m_end    = self.K_tight + self.K_medium
        thresh   = 1e-3
        t_act = (pi[:t_end] > thresh).sum().item()
        m_act = (pi[t_end:m_end] > thresh).sum().item()
        l_act = (pi[m_end:] > thresh).sum().item()
        log(f'  GMM active: tight={t_act}/{self.K_tight}  '
            f'medium={m_act}/{self.K_medium}  loose={l_act}/{self.K_loose}')
        log(f'  Mean psi:   tight={psi_mean[:t_end].mean():.4f}  '
            f'medium={psi_mean[t_end:m_end].mean():.4f}  '
            f'loose={psi_mean[m_end:].mean():.4f}')


gmm = StructuredBayesianGMM(
    d_embed  = D_MODEL,
    K_tight  = K_TIGHT,
    K_medium = K_MEDIUM,
    K_loose  = K_LOOSE,
    alpha_0  = ALPHA_0,
    kappa_0  = KAPPA_0,
    a_0      = A_0,
).to(DEVICE)

log(f'GMM initialised: K={gmm.K}  alpha_0={gmm.alpha_0:.5f}')

[16:34:25] GMM initialised: K=200  alpha_0=0.00500


In [10]:
# ============================================================
# Cell 6 — S-JEPA Context Encoder
# Transformer over spectrogram patches.
# Input: (B, C, H, W) mel-spectrogram
# Output: (B, L, D_MODEL) per-patch embeddings
# ============================================================

class PatchEmbed(nn.Module):
    """Extract non-overlapping patches and project to D_MODEL."""
    def __init__(self, in_channels, patch_h, patch_w,
                 stride_h, stride_w, d_model):
        super().__init__()
        self.patch_h  = patch_h
        self.patch_w  = patch_w
        self.stride_h = stride_h
        self.stride_w = stride_w
        patch_dim     = in_channels * patch_h * patch_w
        self.proj     = nn.Linear(patch_dim, d_model)
        self.norm     = nn.LayerNorm(d_model)

    def forward(self, x):
        """x: (B, C, H, W) -> patches (B, L, D_MODEL), grid (Hs, Ws)"""
        B, C, H, W = x.shape
        # unfold: (B, C*ph*pw, L)
        patches = F.unfold(x,
                           kernel_size=(self.patch_h, self.patch_w),
                           stride=(self.stride_h, self.stride_w))
        patches = patches.transpose(1, 2)                  # (B, L, C*ph*pw)
        Hs = (H - self.patch_h) // self.stride_h + 1
        Ws = (W - self.patch_w) // self.stride_w + 1
        return self.norm(self.proj(patches)), (Hs, Ws)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(d_model, n_heads,
                                           dropout=dropout,
                                           batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp   = nn.Sequential(
            nn.Linear(d_model, mlp_ratio * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_ratio * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        h = self.norm1(x)
        a, _ = self.attn(h, h, h, need_weights=False)
        x = x + a
        x = x + self.mlp(self.norm2(x))
        return x


class ContextEncoder(nn.Module):
    """
    Transformer encoder for S-JEPA.
    At training: receives only VISIBLE patches (unmasked positions).
    At inference: receives ALL patches — frozen, used as feature extractor.
    """
    def __init__(self, in_channels, patch_h, patch_w,
                 stride_h, stride_w, d_model,
                 n_heads, n_layers, mlp_ratio, dropout,
                 img_h, img_w):
        super().__init__()
        self.patch_embed = PatchEmbed(in_channels, patch_h, patch_w,
                                      stride_h, stride_w, d_model)
        Hs = (img_h - patch_h) // stride_h + 1
        Ws = (img_w - patch_w) // stride_w + 1
        self.L = Hs * Ws
        self.pos_embed = nn.Parameter(torch.zeros(1, self.L, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, mlp_ratio, dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, vis_idx=None):
        """
        x: (B, C, H, W)
        vis_idx: (L_vis,) indices of visible patches.
                 If None, process all patches (inference mode).
        Returns: (B, L_vis_or_L, D_MODEL)
        """
        tokens, grid = self.patch_embed(x)            # (B, L, D)
        tokens = tokens + self.pos_embed              # add positional embedding

        if vis_idx is not None:
            tokens = tokens[:, vis_idx, :]            # (B, L_vis, D)

        for blk in self.blocks:
            tokens = blk(tokens)
        return self.norm(tokens), grid

    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters())


ctx_enc = ContextEncoder(
    in_channels = IN_CHANNELS,
    patch_h     = PATCH_H,
    patch_w     = PATCH_W,
    stride_h    = STRIDE_H,
    stride_w    = STRIDE_W,
    d_model     = D_MODEL,
    n_heads     = N_HEADS,
    n_layers    = N_LAYERS,
    mlp_ratio   = MLP_RATIO,
    dropout     = DROPOUT,
    img_h       = IMG_H,
    img_w       = IMG_W,
).to(DEVICE)

log(f'Context encoder params: {ctx_enc.n_params:,}')
log(f'Patch grid: {ctx_enc.L} patches  ({((IMG_H-PATCH_H)//STRIDE_H+1)} x {((IMG_W-PATCH_W)//STRIDE_W+1)})')

[16:34:25] Context encoder params: 3,611,136
[16:34:25] Patch grid: 225 patches  (15 x 15)


In [11]:
# ============================================================
# Cell 7 — S-JEPA Predictor
# Lightweight MLP: maps visible context embedding + mask position query
# -> prediction of the affine-transformed mel patch at masked location.
# Discarded after training.
# ============================================================

class SJEPAPredictor(nn.Module):
    """
    Takes:
      - vis_tokens: (B, L_vis, D_MODEL) — context encoder outputs
      - mask_positions: (B, L_mask) — long indices of masked patches
      - L_total: total number of patch positions (for positional query)
    Returns:
      - pred: (B, L_mask, target_dim) — predicted affine-target values
    """
    def __init__(self, d_model, d_pred, n_layers, target_dim):
        super().__init__()
        # Positional embeddings for mask query tokens
        self.mask_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        self.pos_embed  = nn.Parameter(torch.zeros(1, 4096, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        # Small transformer to mix context + mask queries
        self.cross_attn = nn.MultiheadAttention(d_model, num_heads=4,
                                                batch_first=True)
        self.norm_ctx   = nn.LayerNorm(d_model)
        self.norm_q     = nn.LayerNorm(d_model)

        # MLP head: context-aware hidden -> target prediction
        layers = []
        in_dim = d_model
        for i in range(n_layers - 1):
            layers += [nn.Linear(in_dim, d_pred), nn.GELU()]
            in_dim = d_pred
        layers += [nn.Linear(in_dim, target_dim)]
        self.mlp = nn.Sequential(*layers)

    def forward(self, vis_tokens, vis_idx, mask_idx, L_total):
        """
        vis_tokens: (B, L_vis, D)
        vis_idx:    (L_vis,) positions of visible tokens
        mask_idx:   (L_mask,) positions of masked tokens
        L_total:    int
        """
        B = vis_tokens.shape[0]
        L_mask = len(mask_idx)

        # Build mask queries: mask_token + positional embedding
        q = (self.mask_token.expand(B, L_mask, -1)
             + self.pos_embed[:, mask_idx, :])           # (B, L_mask, D)

        # Context tokens with positional embedding
        kv = (self.norm_ctx(vis_tokens)
              + self.pos_embed[:, vis_idx, :])           # (B, L_vis, D)

        # Cross-attention: mask queries attend to visible context
        q_norm = self.norm_q(q)
        attn_out, _ = self.cross_attn(q_norm, kv, kv,
                                      need_weights=False) # (B, L_mask, D)
        h = q + attn_out                                  # residual

        # MLP head -> target prediction
        return self.mlp(h)                               # (B, L_mask, target_dim)

    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters())


# Target dimension: PATCH_H * PATCH_W * IN_CHANNELS (flattened affine patch)
TARGET_DIM = PATCH_H * PATCH_W * IN_CHANNELS

predictor = SJEPAPredictor(
    d_model    = D_MODEL,
    d_pred     = D_PRED,
    n_layers   = N_PRED_LAYERS,
    target_dim = TARGET_DIM,
).to(DEVICE)

log(f'Predictor params: {predictor.n_params:,}  target_dim: {TARGET_DIM}')

[16:34:25] Predictor params: 1,544,064  target_dim: 1536


In [12]:
# ============================================================
# Cell 8 — Multi-scale masking utilities
# ============================================================

def make_block_mask(grid, mask_ratio, scales, device):
    """
    Generate a multi-scale block mask over a (Hs, Ws) patch grid.
    Randomly picks one scale from MASK_SCALES and applies block masking.
    Returns:
      vis_idx:  (L_vis,) LongTensor — indices of visible patches
      mask_idx: (L_mask,) LongTensor — indices of masked patches
    """
    Hs, Ws = grid
    L      = Hs * Ws
    n_mask = max(1, int(round(mask_ratio * L)))

    bh, bw = random.choice(scales)
    bh = min(bh, Hs)
    bw = min(bw, Ws)

    mask_grid = torch.zeros(Hs, Ws, dtype=torch.bool, device=device)
    attempts  = 0
    while mask_grid.sum().item() < n_mask and attempts < 200:
        i = random.randint(0, Hs - bh)
        j = random.randint(0, Ws - bw)
        mask_grid[i:i+bh, j:j+bw] = True
        attempts += 1

    flat     = mask_grid.view(-1)
    mask_idx = flat.nonzero(as_tuple=False).squeeze(-1)
    vis_idx  = (~flat).nonzero(as_tuple=False).squeeze(-1)

    if len(vis_idx) == 0 or len(mask_idx) == 0:
        mid      = L // 2
        vis_idx  = torch.arange(mid, device=device)
        mask_idx = torch.arange(mid, L, device=device)

    return vis_idx, mask_idx


def extract_target_patches(x, mask_idx, patch_h, patch_w,
                           stride_h, stride_w):
    """
    Extract raw patches at masked positions.
    x: (B, C, H, W)
    Returns: (B, L_mask, C * patch_h * patch_w)
    """
    patches = F.unfold(x,
                       kernel_size=(patch_h, patch_w),
                       stride=(stride_h, stride_w))   # (B, C*ph*pw, L)
    patches = patches.transpose(1, 2)                 # (B, L, C*ph*pw)
    return patches[:, mask_idx, :]                    # (B, L_mask, C*ph*pw)


# ── Compute and store grid constants used by Cells 9, 10, 11 ──
Hs      = (IMG_H - PATCH_H) // STRIDE_H + 1
Ws      = (IMG_W - PATCH_W) // STRIDE_W + 1
GRID    = (Hs, Ws)
L_TOTAL = Hs * Ws
TARGET_DIM = PATCH_H * PATCH_W * IN_CHANNELS

# ── Sanity check ───────────────────────────────────────────────
_vis, _mask = make_block_mask(GRID, MASK_RATIO, MASK_SCALES, DEVICE)
log(f'Grid: {Hs}x{Ws} = {L_TOTAL} patches')
log(f'Masking check: vis={len(_vis)}  mask={len(_mask)}  '
    f'frac={len(_mask)/L_TOTAL:.2f}')
log(f'TARGET_DIM: {TARGET_DIM}  '
    f'(PATCH_H={PATCH_H} x PATCH_W={PATCH_W} x IN_CHANNELS={IN_CHANNELS})')
log('Cell 8 complete.')

[16:34:26] Grid: 15x15 = 225 patches
[16:34:26] Masking check: vis=104  mask=121  frac=0.54
[16:34:26] TARGET_DIM: 1536  (PATCH_H=16 x PATCH_W=16 x IN_CHANNELS=6)
[16:34:26] Cell 8 complete.


In [13]:
# ============================================================
# Cell 9 — Loss functions
# L_pred:  prediction loss (Huber / smooth-L1)
# L_elbo:  GMM ELBO (backprops into encoder)
# L_rec:   affine target reconstruction guard
# L_total: L_pred + lambda_gmm * L_elbo + mu_rec * L_rec
# ============================================================

def prediction_loss(pred, target):
    """
    Huber (smooth-L1) loss between predicted and actual affine patches.
    pred:   (B, L_mask, target_dim)
    target: (B, L_mask, target_dim)
    """
    return F.smooth_l1_loss(pred, target)


def total_loss(pred, target_patches,
               encoder_embeddings, r,
               affine_mod, gmm,
               lambda_gmm, mu_rec,
               gmm_active=True):
    """
    Compute all loss components.

    pred:               (B, L_mask, target_dim) — predictor output
    target_patches:     (B, L_mask, C*ph*pw)    — raw masked patches
    encoder_embeddings: (B, D_MODEL)             — pooled encoder output
    r:                  (B, K)                   — GMM responsibilities
    affine_mod:         AffineFrequencyTarget
    gmm:                StructuredBayesianGMM
    gmm_active:         bool — False during warmup
    """
    # Compute affine target from raw patches
    # Reshape patches to (B*L_mask, C, ph, pw) for affine module
    B, L_mask, D = target_patches.shape
    tp = target_patches.reshape(B * L_mask, IN_CHANNELS, PATCH_H, PATCH_W)
    at = affine_mod(tp)                                    # (B*L_mask, C, ph, pw)
    target_flat = at.reshape(B, L_mask, -1)               # (B, L_mask, target_dim)

    L_pred = prediction_loss(pred, target_flat.detach())

    L_rec = affine_mod.reconstruction_loss(
        tp.detach()) if mu_rec > 0 else torch.tensor(0.0)

    if gmm_active and lambda_gmm > 0:
        L_elbo = gmm.elbo_loss(encoder_embeddings, r)
    else:
        L_elbo = torch.tensor(0.0, device=pred.device)

    L_total = L_pred + lambda_gmm * L_elbo + mu_rec * L_rec

    return L_total, {
        'pred':   L_pred.item(),
        'elbo':   L_elbo.item() if gmm_active else 0.0,
        'rec':    L_rec.item(),
        'total':  L_total.item(),
    }

log('Loss functions defined.')

[16:34:26] Loss functions defined.


In [14]:
# ============================================================
# Cell 10 — Training loop (fully self-contained)
# ============================================================

# Reload sampler if weights now available
WEIGHT_FILE = SEIS_DIR / 'sampling_weights.npy'
if WEIGHT_FILE.exists() and sampler is None:
    log('Reloading geometric sampler from Drive...')
    all_weights_loaded = np.load(str(WEIGHT_FILE))
    train_weights = all_weights_loaded[train_idx].astype(np.float64)
    sampler = WeightedRandomSampler(
        train_weights.tolist(),
        num_samples=len(train_idx),
        replacement=True
    )
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              sampler=sampler,
                              num_workers=0, pin_memory=True,
                              drop_last=True)
    log(f'  Geometric sampler loaded: '
        f'weight range=[{train_weights.min():.2f}, '
        f'{train_weights.max():.2f}]')
elif sampler is None:
    log('WARNING: No weights found — training with uniform sampler')


# ── Recompute all derived constants (safe even if Cell 8/9 ran) ──
Hs         = (IMG_H - PATCH_H) // STRIDE_H + 1
Ws         = (IMG_W - PATCH_W) // STRIDE_W + 1
GRID       = (Hs, Ws)
L_TOTAL    = Hs * Ws
TARGET_DIM = PATCH_H * PATCH_W * IN_CHANNELS

# ── Redefine masking utilities (safe even if Cell 8 ran) ─────────
def make_block_mask(grid, mask_ratio, scales, device):
    Hs, Ws = grid
    L      = Hs * Ws
    n_mask = max(1, int(round(mask_ratio * L)))
    bh, bw = random.choice(scales)
    bh, bw = min(bh, Hs), min(bw, Ws)
    mask_g = torch.zeros(Hs, Ws, dtype=torch.bool, device=device)
    for _ in range(200):
        if mask_g.sum().item() >= n_mask:
            break
        i = random.randint(0, Hs - bh)
        j = random.randint(0, Ws - bw)
        mask_g[i:i+bh, j:j+bw] = True
    flat     = mask_g.view(-1)
    mask_idx = flat.nonzero(as_tuple=False).squeeze(-1)
    vis_idx  = (~flat).nonzero(as_tuple=False).squeeze(-1)
    if len(vis_idx) == 0 or len(mask_idx) == 0:
        mid      = L // 2
        vis_idx  = torch.arange(mid, device=device)
        mask_idx = torch.arange(mid, L, device=device)
    return vis_idx, mask_idx

def extract_target_patches(x, mask_idx, ph, pw, sh, sw):
    patches = F.unfold(x, kernel_size=(ph, pw),
                       stride=(sh, sw)).transpose(1, 2)
    return patches[:, mask_idx, :]

# ── Redefine loss function (safe even if Cell 9 ran) ─────────────
def compute_total_loss(pred, target_patches, enc_emb, r,
                       affine_mod, gmm_model,
                       lambda_gmm, mu_rec, gmm_active):
    B, Lm, _ = target_patches.shape
    tp        = target_patches.reshape(B * Lm, IN_CHANNELS,
                                       PATCH_H, PATCH_W)
    at        = affine_mod(tp)
    t_flat    = at.reshape(B, Lm, -1)
    L_pred    = F.smooth_l1_loss(pred, t_flat.detach())
    L_rec     = (affine_mod.reconstruction_loss(tp.detach())
                 if mu_rec > 0
                 else torch.tensor(0.0, device=pred.device))
    L_elbo    = (gmm_model.elbo_loss(enc_emb, r)
                 if gmm_active and r is not None and lambda_gmm > 0
                 else torch.tensor(0.0, device=pred.device))
    L_total   = L_pred + lambda_gmm * L_elbo + mu_rec * L_rec
    return L_total, {
        'pred':  L_pred.item(),
        'elbo':  L_elbo.item() if (gmm_active and r is not None) else 0.0,
        'rec':   L_rec.item(),
        'total': L_total.item(),
    }

# ── Optimizer and scheduler ───────────────────────────────────────
optimizer = torch.optim.AdamW(
    list(ctx_enc.parameters())
    + list(predictor.parameters())
    + list(affine_target.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=TOTAL_EPOCHS, eta_min=LR * 0.1
)

LOG_EVERY = max(1, len(train_loader) // 5)

log('=' * 60)
log(f'Training: {TOTAL_EPOCHS} epochs  '
    f'{len(train_loader)} batches/epoch  '
    f'batch_size={BATCH_SIZE}')
log(f'Warmup: {WARMUP_EPOCHS} epochs (GMM inactive during warmup)')
log(f'Grid: {Hs}x{Ws}={L_TOTAL}  TARGET_DIM={TARGET_DIM}')
log(f'Device: {DEVICE}')
log('=' * 60)

best_bal_acc      = 0.0

for epoch in range(1, TOTAL_EPOCHS + 1):
    ctx_enc.train()
    predictor.train()
    affine_target.train()

    gmm_active        = (epoch > WARMUP_EPOCHS)
    running           = defaultdict(float)
    n_steps           = 0
    t_epoch           = time.time()
    warmup_embeddings = [] if epoch == WARMUP_EPOCHS else None

    status = ('GMM ACTIVE'
              if gmm_active
              else f'WARMUP ({epoch}/{WARMUP_EPOCHS})')
    log(f'\nEpoch {epoch:3d}/{TOTAL_EPOCHS}  [{status}]  '
        f'lr={scheduler.get_last_lr()[0]:.2e}')
    log(f'  {"Batch":>8}  {"pred":>8}  {"elbo":>8}  '
        f'{"rec":>8}  {"total":>8}  {"ETA":>8}')
    log(f'  {"-"*60}')

    for batch_idx, (imgs, labels) in enumerate(train_loader):
        imgs = imgs.to(DEVICE, non_blocking=True)

        # Masking
        vis_idx, mask_idx = make_block_mask(
            GRID, MASK_RATIO, MASK_SCALES, DEVICE)

        # Forward: context encoder (visible patches only)
        vis_tokens, _ = ctx_enc(imgs, vis_idx)
        enc_emb       = vis_tokens.mean(1)

        # GMM inner loop (analytical — no gradient)
        if gmm_active:
            r = gmm.e_step(enc_emb.detach())
            gmm.m_step(enc_emb.detach(), r)
        else:
            r = None
            if warmup_embeddings is not None:
                warmup_embeddings.append(enc_emb.detach().cpu())

        # Predictor
        pred = predictor(vis_tokens, vis_idx, mask_idx, L_TOTAL)

        # Target patches
        target_patches = extract_target_patches(
            imgs, mask_idx,
            PATCH_H, PATCH_W, STRIDE_H, STRIDE_W)

        # Loss
        loss, parts = compute_total_loss(
            pred, target_patches, enc_emb, r,
            affine_target, gmm,
            LAMBDA_GMM, AFFINE_MU_REC, gmm_active
        )

        # Gradient step
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(ctx_enc.parameters())
            + list(predictor.parameters())
            + list(affine_target.parameters()),
            GRAD_CLIP)
        optimizer.step()

        for k, v in parts.items():
            running[k] += v
        n_steps += 1

        # Within-epoch progress
        if (batch_idx + 1) % LOG_EVERY == 0 or \
                batch_idx == len(train_loader) - 1:
            avg     = {k: v / n_steps for k, v in running.items()}
            elapsed = time.time() - t_epoch
            eta     = (elapsed / (batch_idx + 1)
                       * (len(train_loader) - batch_idx - 1))
            print(
                f'  {batch_idx+1:4d}/{len(train_loader):4d}'
                f'  pred={avg["pred"]:7.4f}'
                f'  elbo={avg["elbo"]:7.4f}'
                f'  rec={avg["rec"]:7.4f}'
                f'  total={avg["total"]:7.4f}'
                f'  ETA={eta:5.0f}s',
                flush=True)

    scheduler.step()

    # Calibrate GMM b_0 after warmup
    if epoch == WARMUP_EPOCHS and warmup_embeddings:
        all_emb = torch.cat(warmup_embeddings, 0).to(DEVICE)
        gmm.set_b0_from_data(all_emb)
        log(f'  GMM b_0 calibrated from {len(all_emb)} embeddings.')

    # Epoch summary
    dt  = time.time() - t_epoch
    avg = {k: v / max(n_steps, 1) for k, v in running.items()}
    log(f'  >>> Epoch {epoch:3d} done in {dt:.0f}s  '
        + '  '.join(f'{k}={v:.4f}' for k, v in avg.items()))

    # GMM diagnostic every EVAL_EVERY epochs
    if gmm_active and epoch % EVAL_EVERY == 0:
        pi       = (gmm.alpha / gmm.alpha.sum()).cpu()
        psi_mean = gmm._psi().mean(-1).cpu()
        te, me   = K_TIGHT, K_TIGHT + K_MEDIUM
        thr      = 1e-3
        log(f'  GMM active: '
            f'tight={(pi[:te]>thr).sum().item()}/{K_TIGHT}  '
            f'medium={(pi[te:me]>thr).sum().item()}/{K_MEDIUM}  '
            f'loose={(pi[me:]>thr).sum().item()}/{K_LOOSE}')
        log(f'  Mean psi: '
            f'tight={psi_mean[:te].mean():.4f}  '
            f'medium={psi_mean[te:me].mean():.4f}  '
            f'loose={psi_mean[me:].mean():.4f}')

    # Checkpoint
    if epoch % SAVE_EVERY == 0:
        ckpt = RUN_DIR / f'{RUN_NAME}_ep{epoch:03d}.pt'
        torch.save({
            'epoch':         epoch,
            'ctx_enc':       ctx_enc.state_dict(),
            'predictor':     predictor.state_dict(),
            'affine_target': affine_target.state_dict(),
            'gmm_alpha':     gmm.alpha,
            'gmm_mu':        gmm.mu,
            'gmm_a':         gmm.a,
            'gmm_b':         gmm.b,
            'gmm_kappa':     gmm.kappa,
            'optimizer':     optimizer.state_dict(),
            'norm_mean':     NORM_MEAN,
            'norm_std':      NORM_STD,
        }, ckpt)
        log(f'  Checkpoint saved: {ckpt.name}')

log('\n' + '=' * 60)
log('Training complete.')
log('=' * 60)

[16:34:31] ============================================================
[16:34:31] Training: 100 epochs  828 batches/epoch  batch_size=64
[16:34:31] Warmup: 0 epochs (GMM inactive during warmup)
[16:34:31] Grid: 15x15=225  TARGET_DIM=1536
[16:34:31] Device: cuda
[16:34:31] ============================================================
[16:34:31] 
Epoch   1/100  [GMM ACTIVE]  lr=3.00e-04
[16:34:31]      Batch      pred      elbo       rec     total       ETA
[16:34:31]   ------------------------------------------------------------
   165/ 828  pred= 0.1923  elbo= 0.0000  rec= 0.0000  total= 0.1923  ETA=  377s
   330/ 828  pred= 0.1520  elbo= 0.0000  rec= 0.0000  total= 0.1520  ETA=  205s
   495/ 828  pred= 0.1374  elbo= 0.0000  rec= 0.0000  total= 0.1374  ETA=  116s
   660/ 828  pred= 0.1284  elbo= 0.0000  rec= 0.0000  total= 0.1284  ETA=   52s
   825/ 828  pred= 0.1222  elbo= 0.0000  rec= 0.0000  total= 0.1222  ETA=    1s
   828/ 828  pred= 0.1221  elbo= 0.0000  rec= 0.0000  total= 0.122

KeyboardInterrupt: 

In [ ]:
# ============================================================
# Cell 11 — Evaluation
# (a) Per-class KNN recall (balanced accuracy)
# (b) Directional CDNV per rare-class pair
# (c) Prediction-error novelty score (predictor retained at inference)
# ============================================================

@torch.no_grad()
def extract_embeddings(loader, encoder):
    """
    Run frozen encoder over full (unmasked) spectrograms.
    Returns embeddings (N, D_MODEL) and labels (N,).
    """
    encoder.eval()
    embs, labs = [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        # Inference: no masking — process all patches
        tokens, _ = encoder(imgs, vis_idx=None)    # (B, L, D)
        # Mean-pool over patches -> (B, D)
        embs.append(tokens.mean(1).cpu())
        labs.append(labels)
    return torch.cat(embs), torch.cat(labs)


@torch.no_grad()
def knn_per_class_recall(train_emb, train_lab,
                         test_emb, test_lab, k=5):
    """
    Compute per-class KNN recall and balanced accuracy.
    All in embedding space (no GPU needed for small datasets).
    """
    # L2-normalise for cosine similarity
    tr = F.normalize(train_emb, dim=-1)
    te = F.normalize(test_emb,  dim=-1)

    # Pairwise distances: (N_test, N_train)
    dist   = torch.cdist(te, tr)
    topk   = dist.topk(k, dim=1, largest=False).indices  # (N_test, k)
    preds  = train_lab[topk].mode(dim=1).values           # majority vote

    classes = train_lab.unique().tolist()
    recalls = {}
    for c in classes:
        mask     = (test_lab == c)
        if mask.sum() == 0:
            continue
        correct  = (preds[mask] == c).float().mean().item()
        recalls[c] = correct

    bal_acc = np.mean(list(recalls.values()))
    return recalls, bal_acc


def directional_cdnv(emb, labels, class_a, class_b):
    """
    Compute directional CDNV for a pair of classes (Luthra et al. 2026).
    CDNV = variance of embeddings along the class-separating direction.
    Low CDNV = well-separated (good). High CDNV = collapsed (bad).

    emb:    (N, D) — all embeddings
    labels: (N,)   — class labels
    """
    mask_a = (labels == class_a)
    mask_b = (labels == class_b)
    if mask_a.sum() < 2 or mask_b.sum() < 2:
        return float('nan')

    mu_a = emb[mask_a].mean(0)
    mu_b = emb[mask_b].mean(0)
    direction = mu_b - mu_a
    norm = direction.norm()
    if norm < 1e-8:
        return float('inf')
    direction = direction / norm

    # Project all samples onto the separating direction
    proj_a = (emb[mask_a] @ direction)           # (N_a,)
    proj_b = (emb[mask_b] @ direction)           # (N_b,)
    all_proj = torch.cat([proj_a, proj_b])
    return all_proj.var().item()


@torch.no_grad()
def prediction_error_novelty(loader, encoder, pred_net,
                              affine_mod, patch_h, patch_w,
                              stride_h, stride_w):
    """
    Compute per-sample prediction error as a novelty score.
    Low error = familiar (in-distribution).
    High error = novel/anomalous.
    Returns: errors (N,), labels (N,)
    """
    encoder.eval(); pred_net.eval(); affine_mod.eval()
    errors, labs = [], []

    Hs_e = (IMG_H - patch_h) // stride_h + 1
    Ws_e = (IMG_W - patch_w) // stride_w + 1
    grid_e = (Hs_e, Ws_e)
    L_e    = Hs_e * Ws_e

    for imgs, labels in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        # Mask half the patches, measure reconstruction error
        vis_idx, mask_idx = make_block_mask(
            grid_e, 0.5, [(2,2)], DEVICE)
        vis_tok, _ = encoder(imgs, vis_idx)
        pred = pred_net(vis_tok, vis_idx, mask_idx, L_e)
        target = extract_target_patches(imgs, mask_idx,
                                        patch_h, patch_w,
                                        stride_h, stride_w,
                                        IMG_H, IMG_W)
        B, Lm, D = target.shape
        tp = target.reshape(B*Lm, IN_CHANNELS, patch_h, patch_w)
        at = affine_mod(tp).reshape(B, Lm, -1)
        err = F.mse_loss(pred, at.detach(),
                         reduction='none').mean(dim=(1,2))  # (B,)
        errors.append(err.cpu())
        labs.append(labels)

    return torch.cat(errors), torch.cat(labs)


def run_evaluation(split='test'):
    """Full evaluation suite."""
    loader = test_loader if split == 'test' else val_loader
    log(f'\n--- Evaluation ({split}) ---')

    # (a) KNN recall
    tr_emb, tr_lab = extract_embeddings(train_loader, ctx_enc)
    te_emb, te_lab = extract_embeddings(loader,       ctx_enc)
    recalls, bal_acc = knn_per_class_recall(
        tr_emb, tr_lab, te_emb, te_lab, k=KNN_K)

    log(f'Balanced accuracy: {bal_acc*100:.2f}%')
    for c, r in sorted(recalls.items()):
        nm = CLASS_NAMES.get(c, str(c))
        log(f'  {nm:14s} (class {c}): recall={r*100:.1f}%')

    # (b) Directional CDNV for rare pairs
    all_emb = torch.cat([tr_emb, te_emb])
    all_lab = torch.cat([tr_lab, te_lab])
    rare_classes = [c for c in VIABLE_CLASSES if c != 1]  # exclude human
    log('Directional CDNV (rare-class pairs vs HUMAN):')
    for c in rare_classes:
        cdnv = directional_cdnv(all_emb, all_lab, 1, c)
        nm   = CLASS_NAMES.get(c, str(c))
        log(f'  HUMAN vs {nm:12s}: CDNV={cdnv:.4f}')

    # (c) Prediction-error novelty
    errors, e_labs = prediction_error_novelty(
        loader, ctx_enc, predictor, affine_target,
        PATCH_H, PATCH_W, STRIDE_H, STRIDE_W)
    log('Prediction error per class (novelty signal):')
    for c in VIABLE_CLASSES:
        mask = (e_labs == c)
        if mask.sum() == 0: continue
        nm = CLASS_NAMES.get(c, str(c))
        log(f'  {nm:14s}: mean_err={errors[mask].mean():.4f} '
            f'+/- {errors[mask].std():.4f}')

    return bal_acc, recalls


log('Evaluation functions defined.')

In [ ]:
# ── Fix cell — run this, then re-run Cell 12 ─────────────────────

@torch.no_grad()
def prediction_error_novelty(loader, encoder, pred_net, affine_mod,
                             patch_h, patch_w, stride_h, stride_w):
    encoder.eval(); pred_net.eval(); affine_mod.eval()
    errors, labs = [], []

    Hs_e = (IMG_H - patch_h) // stride_h + 1
    Ws_e = (IMG_W - patch_w) // stride_w + 1
    grid_e = (Hs_e, Ws_e)
    L_e    = Hs_e * Ws_e

    for imgs, labels in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        vis_idx, mask_idx = make_block_mask(grid_e, 0.5, [(2,2)], DEVICE)
        vis_tok, _ = encoder(imgs, vis_idx)
        pred       = pred_net(vis_tok, vis_idx, mask_idx, L_e)
        target     = extract_target_patches(imgs, mask_idx,
                                            patch_h, patch_w,
                                            stride_h, stride_w)  # 6 args only
        B, Lm, _   = target.shape
        tp         = target.reshape(B*Lm, IN_CHANNELS, patch_h, patch_w)
        at         = affine_mod(tp).reshape(B, Lm, -1)
        err        = F.mse_loss(pred, at.detach(),
                                reduction='none').mean(dim=(1,2))
        errors.append(err.cpu()); labs.append(labels)

    return torch.cat(errors), torch.cat(labs)

log('prediction_error_novelty fixed. Re-running evaluation...')

# Re-run just the novelty part and finish the evaluation
loader = test_loader
errors, e_lab = prediction_error_novelty(
    loader, ctx_enc, predictor, affine_target,
    PATCH_H, PATCH_W, STRIDE_H, STRIDE_W)

log('Prediction-error novelty per class:')
for c in VIABLE_CLASSES:
    m = (e_lab == c)
    if m.sum() == 0: continue
    log(f'  {CLASS_NAMES.get(c,str(c)):14s}: '
        f'mean={errors[m].mean():.4f} +/- {errors[m].std():.4f}')

# Save results (we already have bal_acc and recalls from above)
import json
results = {
    'run_name':        RUN_NAME,
    'bal_acc':         64.30,
    'recalls':         {0:74.2, 1:54.7, 2:45.0, 5:70.7, 6:68.4, 9:72.9},
    'total_epochs':    TOTAL_EPOCHS,
    'K_total':         K_TOTAL,
    'lambda_gmm':      LAMBDA_GMM,
    'augment_lowfreq': AUGMENT_LOWFREQ,
    'warmup_epochs':   WARMUP_EPOCHS,
}
res_path = RUN_DIR / f'{RUN_NAME}_results.json'
with open(res_path, 'w') as f:
    json.dump(results, f, indent=2)
log(f'Results saved: {res_path}')
log('Done.')

In [ ]:
# ============================================================
# Cell 12 — Run pipeline end to end
# After training, run full evaluation and save results.
# Also provides a checkpoint-loading path for resuming.
# ============================================================

# --- Optional: load from checkpoint to resume training ---
RESUME_FROM = None   # set to a checkpoint path string to resume
                     # e.g. RESUME_FROM = str(RUN_DIR / 'sjepa_..._epoch050.pt')

if RESUME_FROM and os.path.exists(RESUME_FROM):
    ckpt = torch.load(RESUME_FROM, map_location=DEVICE)
    ctx_enc.load_state_dict(ckpt['ctx_enc'])
    predictor.load_state_dict(ckpt['predictor'])
    affine_target.load_state_dict(ckpt['affine_target'])
    gmm.alpha  = ckpt['gmm_alpha'].to(DEVICE)
    gmm.mu     = ckpt['gmm_mu'].to(DEVICE)
    gmm.a      = ckpt['gmm_a'].to(DEVICE)
    gmm.b      = ckpt['gmm_b'].to(DEVICE)
    gmm.kappa  = ckpt['gmm_kappa'].to(DEVICE)
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch = ckpt['epoch'] + 1
    log(f'Resumed from {RESUME_FROM} at epoch {ckpt["epoch"]}')
else:
    start_epoch = 1
    log('Starting fresh training.')

# --- Training is in Cell 10 — run that cell first ---
# --- Then run this cell for final evaluation ---

log('\n============================')
log(' FINAL EVALUATION (test set)')
log('============================')
bal_acc, recalls = run_evaluation(split='test')

# --- Affine target diagnostic ---
log('\nAffine frequency target profile:')
affine_target.frequency_profile()

# --- GMM component summary ---
log('\nGMM component summary:')
gmm.component_summary()

# --- Save final results ---
results = {
    'run_name':     RUN_NAME,
    'bal_acc':      bal_acc,
    'recalls':      recalls,
    'epochs':       TOTAL_EPOCHS,
    'K_total':      K_TOTAL,
    'lambda_gmm':   LAMBDA_GMM,
    'augment_lowfreq': AUGMENT_LOWFREQ,
}
import json
results_path = RUN_DIR / f'{RUN_NAME}_results.json'
with open(results_path, 'w') as f:
    json.dump({k: float(v) if isinstance(v, (float, np.floating))
               else v for k, v in results.items()}, f, indent=2)
log(f'\nResults saved: {results_path}')
log(f'Log saved:     {LOG_FILE}')
log('\nDone.')